In [4]:
import json
import time
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# 1. Load the Corpus from Phase 1
with open("nccn_parsed_corpus.json", "r", encoding="utf-8") as f:
    saved_chunks = json.load(f)

# We'll use the first 100 chunks for a snappy benchmark
test_corpus = [chunk["text"] for chunk in saved_chunks[:100]] 

print(f"Loaded {len(test_corpus)} medical chunks for testing.")

# 2. Define Clinical Test Queries
test_queries = [
    "What is the management strategy for a patient with a BARD1 mutation?", 
    "Screening recommendations for a proband with family history of breast cancer", 
    "What is the absolute risk of pancreatic cancer for an ATM mutation?" 
]

# 3. Initialize Models to Benchmark
embedding_models = {
    "MiniLM (Baseline)": "all-MiniLM-L6-v2",
    "BGE-Small (Advanced Dense)": "BAAI/bge-small-en-v1.5",
    "PubMedBERT (Medical)": "pritamdeka/S-PubMedBert-MS-MARCO"
}

benchmark_results = []

print("\nStarting Embedding Benchmark...")

for model_name, model_path in embedding_models.items():
    print(f"-> Encoding with {model_name}...")
    
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer(model_path, device=device)
    
    # BGE models perform better with a specific instruction prefix for queries
    query_prefix = "Represent this sentence for searching relevant passages: " if "bge" in model_path else ""
    
    start_time = time.time()
    
    # Encode the medical chunks
    corpus_embeddings = model.encode(test_corpus, convert_to_tensor=True)
    
    # Encode the queries
    queries_to_embed = [query_prefix + q for q in test_queries]
    query_embeddings = model.encode(queries_to_embed, convert_to_tensor=True)
    
    # Calculate Cosine Similarity
    hits = util.semantic_search(query_embeddings, corpus_embeddings, top_k=1)
    
    execution_time = time.time() - start_time
    
    # Record the metrics
    metrics = {
        "Model": model_name,
        "Encoding Time (s)": round(execution_time, 2),
        "Q1 Top Score": round(hits[0][0]['score'], 3),
        "Q2 Top Score": round(hits[1][0]['score'], 3),
        "Q3 Top Score": round(hits[2][0]['score'], 3)
    }
    benchmark_results.append(metrics)

# Generate the Benchmark Report Deliverable
df_embeddings = pd.DataFrame(benchmark_results)
print("\n--- EMBEDDING BENCHMARK REPORT ---")
display(df_embeddings)

# Preview the top retrieval from the Medical Model
print("\n--- TOP RETRIEVAL PREVIEW (PubMedBERT) ---")
best_chunk_index = hits[0][0]['corpus_id']
print(f"Query: {test_queries[0]}")
print(f"Retrieved Context:\n{test_corpus[best_chunk_index]}...")

Loaded 100 medical chunks for testing.

Starting Embedding Benchmark...
-> Encoding with MiniLM (Baseline)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

-> Encoding with BGE-Small (Advanced Dense)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

-> Encoding with PubMedBERT (Medical)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


--- EMBEDDING BENCHMARK REPORT ---


,Model,Encoding Time (s),Q1 Top Score,Q2 Top Score,Q3 Top Score
0,MiniLM (Baseline),0.17,0.417,0.600,0.642
1,BGE-Small (Advanced Dense),0.48,0.659,0.757,0.848
2,PubMedBERT (Medical),1.05,0.893,0.936,0.948



--- TOP RETRIEVAL PREVIEW (PubMedBERT) ---
Query: What is the management strategy for a patient with a BARD1 mutation?
Retrieved Context:
BRCA2 P/LP variants in whom the lifetime risk of breast cancer is up to 7% ,... Footnotes  Footnote g revised: There are only limited data to support screening for male breast cancer. Because of lack of screening, males diagnosed with breast cancer have historically presented with advanced stage disease. There are limited data describing the performance of breast screening for males at inherited risk, however recent studies suggest that the detection rate is similar or better than for females at population risk. Gao Y, et al Radiology 2019;293:282-291 ; Li S, et al. J Clin Oncol 2022;40:1529-1541.  Footnote h added: In males, breast cancer risk in BRCA1 carriers is lower than that in BRCA2 carriers. UPDATES...


In [3]:
import json

print(json.dumps(hits, indent=4))

[
    [
        {
            "corpus_id": 25,
            "score": 0.8927595615386963
        }
    ],
    [
        {
            "corpus_id": 33,
            "score": 0.9357724785804749
        }
    ],
    [
        {
            "corpus_id": 48,
            "score": 0.9477498531341553
        }
    ]
]
